# Cardiovascular Disease: Data Cleaning and EDA

This notebook prepares the cardiovascular disease dataset for modelling and creates exploratory data analysis evidence.

Research question: **To what extent can routine health indicators be used to predict whether an individual has cardiovascular disease?**

Use case: a learning prototype for cardiovascular risk screening. The model later in the project should support prioritisation and explanation, not medical diagnosis.

This notebook does **not** train a model. It focuses on data preparation, cleaning decisions, and exploratory analysis.

## Notebook Outputs

When run successfully, this notebook creates:

- cleaned dataset: `project_1_cardiovascular_disease/01_data/processed/cardio_cleaned.csv`
- cleaning summary table: `project_1_cardiovascular_disease/04_outputs/tables/cardio_cleaning_summary.csv`
- EDA summary table: `project_1_cardiovascular_disease/04_outputs/tables/cardio_eda_group_rate_summary.csv`
- EDA charts: `project_1_cardiovascular_disease/04_outputs/charts/`

Run the notebook from top to bottom in VS Code so that each step follows the evidence trail.

## 1. Setup

This cell sets paths, imports libraries, and creates output folders. It is the only setup cell needed for the rest of the notebook.

In [ ]:
import os
import tempfile
from pathlib import Path

# Keep Matplotlib cache files out of the user home directory when running locally.
os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'm5_matplotlib_cache'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)
pd.set_option('display.float_format', '{:,.3f}'.format)

RANDOM_STATE = 42


def find_workspace_root() -> Path:
    """Find the portfolio workspace root from the current notebook location."""
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'project_1_cardiovascular_disease').exists():
            return candidate
    raise FileNotFoundError('Could not find project_1_cardiovascular_disease from the current working directory.')


WORKSPACE_ROOT = find_workspace_root()
PROJECT_DIR = WORKSPACE_ROOT / 'project_1_cardiovascular_disease'
RAW_PATH = PROJECT_DIR / '01_data' / 'raw' / 'cardio_data.csv'
PROCESSED_DIR = PROJECT_DIR / '01_data' / 'processed'
CHARTS_DIR = PROJECT_DIR / '04_outputs' / 'charts'
TABLES_DIR = PROJECT_DIR / '04_outputs' / 'tables'
PROCESSED_PATH = PROCESSED_DIR / 'cardio_cleaned.csv'

for folder in [PROCESSED_DIR, CHARTS_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f'Workspace root: {WORKSPACE_ROOT}')
print(f'Raw dataset path: {RAW_PATH}')
print(f'Processed output path: {PROCESSED_PATH}')

## 2. Load the Raw Dataset

The raw CSV uses a semicolon delimiter. This cell loads the dataset exactly as provided before any cleaning.

In [ ]:
expected_columns = [
    'id', 'age', 'gender', 'height', 'weight', 'ap_hi', 'ap_lo',
    'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'cardio'
]

if not RAW_PATH.exists():
    raise FileNotFoundError(f'Raw data file not found: {RAW_PATH}')

raw_df = pd.read_csv(RAW_PATH, sep=';')

missing_columns = sorted(set(expected_columns) - set(raw_df.columns))
extra_columns = sorted(set(raw_df.columns) - set(expected_columns))

if missing_columns:
    raise ValueError(f'Missing expected columns: {missing_columns}')

print(f'Raw dataset shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]:,} columns')
print(f'Extra columns not expected: {extra_columns if extra_columns else "None"}')
display(raw_df.head())

## 3. Initial Structure Checks

Before cleaning, confirm data types, missing values, unique values, and target balance.

In [ ]:
structure_summary = pd.DataFrame({
    'dtype': raw_df.dtypes.astype(str),
    'missing_values': raw_df.isna().sum(),
    'missing_rate': raw_df.isna().mean(),
    'unique_values': raw_df.nunique(dropna=False),
})

target_summary = (
    raw_df['cardio']
    .value_counts(dropna=False)
    .rename_axis('cardio')
    .reset_index(name='records')
    .sort_values('cardio')
)
target_summary['share'] = target_summary['records'] / len(raw_df)

display(structure_summary)
display(target_summary)

## 4. Duplicate Checks

The dataset has an `id` column, so two checks are useful:

- full-row duplicates, where every column is identical
- analytical duplicates, where all analysis fields match but `id` is different

Analytical duplicates are checked and reported, but not automatically removed here because they may represent different individuals with the same recorded measurements.

In [ ]:
analysis_columns = [column for column in raw_df.columns if column != 'id']

duplicate_summary = pd.DataFrame([
    {
        'check': 'Full-row duplicates',
        'records_flagged': int(raw_df.duplicated().sum()),
        'action': 'Remove only if any are present',
    },
    {
        'check': 'Duplicates excluding id',
        'records_flagged': int(raw_df.duplicated(subset=analysis_columns).sum()),
        'action': 'Report only; keep because records may be different individuals',
    },
])

display(duplicate_summary)

## 5. Cleaning Rules

Cleaning rules are deliberately limited to source measurements that look unsuitable for a cardiovascular screening model. Abnormal health values are **not** automatically removed just because they are unusual.

The updated approach separates two ideas:

- **data quality rule**: remove values that look impossible or highly likely to be recording errors
- **clinical risk category**: keep valid but high-risk values, then describe them in EDA

| Field | Rule | Reason |
|---|---:|---|
| `age` | convert from days to years | easier to interpret and model |
| `height` | keep 100-260 cm | broad adult plausibility range; keeps very tall adults and removes impossible-looking entries such as 55-97 cm |
| `weight` | keep 25-300 kg | broad adult plausibility range; keeps very high weights and removes impossible-looking entries such as 10-23 kg |
| `ap_hi` | keep 50-300 mmHg | keeps clinically extreme readings while removing obvious recording errors such as negative or four/five-digit values |
| `ap_lo` | keep 30-200 mmHg | keeps clinically extreme readings while removing obvious recording errors such as negative or four/five-digit values |
| blood pressure order | keep where systolic >= diastolic | the first blood pressure number should be higher than the second |
| BMI | create from height and weight; do not use as a deletion rule | high BMI can be medically possible, so it should be retained when source height and weight are plausible |

References used for interpretation:

- NHS BMI guidance: BMI is a screening measure and is not suitable for every individual context, especially where height is affected by a condition.
- CDC adult BMI categories: BMI 40 or above is class 3/severe obesity, not an automatic data error.
- NHS blood pressure guidance: blood pressure interpretation should distinguish low, normal, and high readings; abnormal readings are not automatically invalid.

The aim is not to make the data look perfect. The aim is to remove records that are unlikely to be valid enough for modelling while preserving medically possible high-risk values.

## 6. Engineer Core Fields and Flag Quality Issues

This cell creates interpretable fields and counts how many records would be removed by each cleaning rule.

In [ ]:
def add_engineered_fields(df: pd.DataFrame) -> pd.DataFrame:
    """Create interpretable fields used for cleaning, EDA, and later modelling."""
    output = df.copy()
    output['age_years'] = output['age'] / 365.25
    output['height_m'] = output['height'] / 100
    output['bmi'] = output['weight'] / (output['height_m'] ** 2)
    output['pulse_pressure'] = output['ap_hi'] - output['ap_lo']
    return output


def build_quality_flags(df: pd.DataFrame) -> pd.DataFrame:
    """Return boolean flags where True means the record fails a cleaning rule."""
    flags = pd.DataFrame(index=df.index)
    flags['height_out_of_range'] = ~df['height'].between(100, 260, inclusive='both')
    flags['weight_out_of_range'] = ~df['weight'].between(25, 300, inclusive='both')
    flags['systolic_bp_out_of_range'] = ~df['ap_hi'].between(50, 300, inclusive='both')
    flags['diastolic_bp_out_of_range'] = ~df['ap_lo'].between(30, 200, inclusive='both')
    flags['systolic_less_than_diastolic'] = df['ap_hi'] < df['ap_lo']
    flags['any_quality_issue'] = flags.any(axis=1)
    return flags


work_df = add_engineered_fields(raw_df)
quality_flags = build_quality_flags(work_df)

quality_summary = (
    quality_flags
    .sum()
    .rename('records_flagged')
    .reset_index()
    .rename(columns={'index': 'quality_check'})
)
quality_summary['share_of_raw_data'] = quality_summary['records_flagged'] / len(work_df)

display(quality_summary)

## 7. Apply Cleaning Rules

Records with at least one serious quality issue are removed. The target distribution is checked before and after cleaning to make sure the cleaning does not heavily distort the outcome balance.

In [ ]:
clean_df = work_df.loc[~quality_flags['any_quality_issue']].copy()
removed_df = work_df.loc[quality_flags['any_quality_issue']].copy()

cleaning_summary = pd.DataFrame([
    {'stage': 'Raw data', 'records': len(raw_df), 'removed_from_previous_stage': 0},
    {'stage': 'After quality-rule cleaning', 'records': len(clean_df), 'removed_from_previous_stage': len(removed_df)},
])
cleaning_summary['share_of_raw_data'] = cleaning_summary['records'] / len(raw_df)

before_after_target = pd.concat(
    [
        raw_df['cardio'].value_counts(normalize=True).sort_index().rename('before_cleaning'),
        clean_df['cardio'].value_counts(normalize=True).sort_index().rename('after_cleaning'),
    ],
    axis=1,
).rename_axis('cardio')

cleaning_summary.to_csv(TABLES_DIR / 'cardio_cleaning_summary.csv', index=False)
before_after_target.to_csv(TABLES_DIR / 'cardio_target_balance_before_after_cleaning.csv')

display(cleaning_summary)
display(before_after_target)
print(f'Cleaning summary saved to: {TABLES_DIR / "cardio_cleaning_summary.csv"}')

## 8. Create Readable Labels and Ordered Bands

These fields make the EDA charts easier to read. The original numeric fields are kept for modelling.

In [ ]:
clean_df = clean_df.rename(columns={
    'age': 'age_days',
    'height': 'height_cm',
    'weight': 'weight_kg',
})

cholesterol_labels = {
    1: 'Normal',
    2: 'Above normal',
    3: 'Well above normal',
}

glucose_labels = {
    1: 'Normal',
    2: 'Above normal',
    3: 'Well above normal',
}

binary_labels = {0: 'No', 1: 'Yes'}
cardio_labels = {0: 'No cardiovascular disease', 1: 'Cardiovascular disease'}

clean_df['cholesterol_label'] = clean_df['cholesterol'].map(cholesterol_labels)
clean_df['glucose_label'] = clean_df['gluc'].map(glucose_labels)
clean_df['smoke_label'] = clean_df['smoke'].map(binary_labels)
clean_df['alcohol_label'] = clean_df['alco'].map(binary_labels)
clean_df['active_label'] = clean_df['active'].map(binary_labels)
clean_df['cardio_label'] = clean_df['cardio'].map(cardio_labels)
clean_df['gender_label'] = clean_df['gender'].map({1: 'Gender code 1', 2: 'Gender code 2'})

clean_df['age_band'] = pd.cut(
    clean_df['age_years'],
    bins=[0, 40, 50, 60, 120],
    labels=['Under 40', '40-49', '50-59', '60+'],
    right=False,
)

clean_df['bmi_category'] = pd.cut(
    clean_df['bmi'],
    bins=[0, 18.5, 25, 30, 40, np.inf],
    labels=['Underweight', 'Healthy weight', 'Overweight', 'Obesity', 'Severe obesity'],
    right=False,
)

clean_df['systolic_bp_category'] = pd.cut(
    clean_df['ap_hi'],
    bins=[0, 120, 130, 140, 160, 300],
    labels=['<120', '120-129', '130-139', '140-159', '160+'],
    right=False,
)

clean_df['diastolic_bp_category'] = pd.cut(
    clean_df['ap_lo'],
    bins=[0, 80, 90, 100, 110, 200],
    labels=['<80', '80-89', '90-99', '100-109', '110+'],
    right=False,
)

label_checks = clean_df[
    ['age_band', 'bmi_category', 'systolic_bp_category', 'diastolic_bp_category', 'cholesterol_label', 'glucose_label']
].isna().sum().rename('missing_labels').to_frame()

display(label_checks)
display(clean_df.head())

## 9. Save the Cleaned Dataset

The saved file drops `id` because it is an identifier, not a meaningful health indicator for modelling.

In [ ]:
cleaned_columns = [
    'age_days', 'age_years', 'gender', 'gender_label',
    'height_cm', 'weight_kg', 'bmi',
    'ap_hi', 'ap_lo', 'pulse_pressure',
    'cholesterol', 'cholesterol_label',
    'gluc', 'glucose_label',
    'smoke', 'smoke_label',
    'alco', 'alcohol_label',
    'active', 'active_label',
    'age_band', 'bmi_category', 'systolic_bp_category', 'diastolic_bp_category',
    'cardio', 'cardio_label',
]

clean_df_out = clean_df[cleaned_columns].copy()
clean_df_out.to_csv(PROCESSED_PATH, index=False)

print(f'Cleaned dataset saved to: {PROCESSED_PATH}')
print(f'Cleaned dataset shape: {clean_df_out.shape[0]:,} rows x {clean_df_out.shape[1]:,} columns')
display(clean_df_out.head())

## 10. EDA Helper Functions

The charts use percentage rates rather than raw counts where the question is about cardiovascular disease likelihood within groups.

In [ ]:
plt.rcParams.update({
    'figure.figsize': (9, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'bold',
    'axes.labelsize': 10,
    'axes.titlesize': 13,
    'font.size': 10,
})

BAR_COLOUR = '#2F4858'
HIGHLIGHT_COLOUR = '#F26419'
LINE_COLOUR = '#44546A'
GRID_COLOUR = '#D9DEE7'


def save_chart(fig: plt.Figure, filename: str) -> Path:
    output_path = CHARTS_DIR / filename
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches='tight')
    return output_path


def group_rate_table(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    table = (
        df.groupby(group_col, observed=False)
        .agg(records=('cardio', 'size'), cardiovascular_disease_rate=('cardio', 'mean'))
        .reset_index()
    )
    return table


def plot_rate_by_group(df: pd.DataFrame, group_col: str, title: str, xlabel: str, filename: str) -> pd.DataFrame:
    table = group_rate_table(df, group_col)
    overall_rate = df['cardio'].mean()

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.set_axisbelow(True)
    bars = ax.bar(
        table[group_col].astype(str),
        table['cardiovascular_disease_rate'],
        color=BAR_COLOUR,
        zorder=3,
    )

    ax.axhline(overall_rate, color=HIGHLIGHT_COLOUR, linestyle='--', linewidth=1.2, alpha=0.7, zorder=1)
    ax.text(
        len(table) - 0.55,
        overall_rate + 0.01,
        f'Overall rate: {overall_rate:.1%}',
        color=HIGHLIGHT_COLOUR,
        ha='right',
        va='bottom',
        fontsize=9,
    )

    for bar, rate in zip(bars, table['cardiovascular_disease_rate']):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            rate + 0.01,
            f'{rate:.1%}',
            ha='center',
            va='bottom',
            fontsize=9,
        )

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Cardiovascular disease rate')
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_ylim(0, min(1.0, max(table['cardiovascular_disease_rate'].max() + 0.12, overall_rate + 0.12)))
    ax.grid(axis='y', color=GRID_COLOUR, linewidth=0.45, alpha=0.25, linestyle=(0, (4, 6)), zorder=0)

    output_path = save_chart(fig, filename)
    plt.show()
    print(f'Saved chart: {output_path}')
    return table

## 11. Target Distribution

This chart confirms whether the outcome is balanced enough for a straightforward binary classification workflow.

In [ ]:
target_order = ['No cardiovascular disease', 'Cardiovascular disease']
target_counts = clean_df_out['cardio_label'].value_counts().reindex(target_order)
target_share = target_counts / target_counts.sum()

target_table = pd.DataFrame({
    'cardio_label': target_counts.index,
    'records': target_counts.values,
    'share': target_share.values,
})

fig, ax = plt.subplots(figsize=(8, 5))
ax.set_axisbelow(True)
bars = ax.bar(target_table['cardio_label'], target_table['records'], width=0.46, color=[BAR_COLOUR, '#0F766E'], edgecolor='none', zorder=3)

for bar, records, share in zip(bars, target_table['records'], target_table['share']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        records + 500,
        f'{records:,.0f}\n({share:.1%})',
        ha='center',
        va='bottom',
        fontsize=10,
    )

ax.set_title('Target distribution after cleaning')
ax.set_xlabel('Cardiovascular disease status')
ax.set_ylabel('Number of records')
ax.grid(axis='y', color=GRID_COLOUR, linewidth=0.45, alpha=0.25, linestyle=(0, (4, 6)), zorder=0)
ax.set_ylim(0, target_table['records'].max() * 1.15)

output_path = save_chart(fig, 'eda_01_target_distribution_after_cleaning.png')
plt.show()
print(f'Saved chart: {output_path}')
display(target_table)

## 12. Cardiovascular Disease Rate by Health Indicator

These charts show how the disease rate changes across interpretable groups. They are not causal proof, but they help explain which fields may be useful predictors later.

In [ ]:
rate_tables = []

chart_specs = [
    ('age_band', 'Cardiovascular disease rate by age band', 'Age band', 'eda_02_rate_by_age_band.png'),
    ('systolic_bp_category', 'Cardiovascular disease rate by systolic blood pressure category', 'Systolic blood pressure category (mmHg)', 'eda_03_rate_by_systolic_bp_category.png'),
    ('diastolic_bp_category', 'Cardiovascular disease rate by diastolic blood pressure category', 'Diastolic blood pressure category (mmHg)', 'eda_04_rate_by_diastolic_bp_category.png'),
    ('bmi_category', 'Cardiovascular disease rate by BMI category', 'BMI category', 'eda_05_rate_by_bmi_category.png'),
    ('cholesterol_label', 'Cardiovascular disease rate by cholesterol level', 'Cholesterol level', 'eda_06_rate_by_cholesterol.png'),
    ('glucose_label', 'Cardiovascular disease rate by glucose level', 'Glucose level', 'eda_07_rate_by_glucose.png'),
    ('active_label', 'Cardiovascular disease rate by physical activity status', 'Physically active', 'eda_08_rate_by_activity.png'),
]

for group_col, title, xlabel, filename in chart_specs:
    table = plot_rate_by_group(clean_df_out, group_col, title, xlabel, filename)
    table.insert(0, 'feature', group_col)
    table = table.rename(columns={group_col: 'group'})
    rate_tables.append(table)

## 13. Correlation Check

This heatmap gives a quick view of linear relationships between numeric fields. It is useful before logistic regression because highly related variables can make interpretation harder.

In [ ]:
numeric_columns = [
    'cardio', 'age_years', 'height_cm', 'weight_kg', 'bmi',
    'ap_hi', 'ap_lo', 'pulse_pressure',
    'cholesterol', 'gluc', 'smoke', 'alco', 'active'
]

corr = clean_df_out[numeric_columns].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)

ax.set_xticks(range(len(numeric_columns)))
ax.set_yticks(range(len(numeric_columns)))
ax.set_xticklabels(numeric_columns, rotation=45, ha='right')
ax.set_yticklabels(numeric_columns)
ax.set_title('Correlation heatmap for numeric fields')

for row_index in range(len(numeric_columns)):
    for column_index in range(len(numeric_columns)):
        value = corr.iloc[row_index, column_index]
        text_colour = 'white' if abs(value) > 0.55 else 'black'
        ax.text(column_index, row_index, f'{value:.2f}', ha='center', va='center', color=text_colour, fontsize=8)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Correlation')

output_path = save_chart(fig, 'eda_09_numeric_correlation_heatmap.png')
plt.show()
print(f'Saved chart: {output_path}')

## 14. EDA Summary Table

This table stores the grouped cardiovascular disease rates used in the charts. It can be referenced later in the report or portfolio write-up.

In [ ]:
eda_group_rate_summary = pd.concat(rate_tables, ignore_index=True)
eda_group_rate_summary['cardiovascular_disease_rate'] = eda_group_rate_summary['cardiovascular_disease_rate'].round(4)

eda_summary_path = TABLES_DIR / 'cardio_eda_group_rate_summary.csv'
eda_group_rate_summary.to_csv(eda_summary_path, index=False)

signal_strength = (
    eda_group_rate_summary
    .groupby('feature')
    .agg(
        lowest_group_rate=('cardiovascular_disease_rate', 'min'),
        highest_group_rate=('cardiovascular_disease_rate', 'max'),
    )
    .reset_index()
)
signal_strength['rate_range'] = signal_strength['highest_group_rate'] - signal_strength['lowest_group_rate']
signal_strength = signal_strength.sort_values('rate_range', ascending=False)

print(f'EDA group-rate summary saved to: {eda_summary_path}')
display(eda_group_rate_summary)
display(signal_strength)

## 15. Interim EDA Notes

Evidence from this notebook should be used carefully:

- The target is binary, so logistic regression is a suitable first modelling method.
- The target remains close to balanced after cleaning, so accuracy can be reported, but recall is still important for the screening-support use case.
- The grouped charts show associations, not causes.
- The dataset does not include medical history, medication, symptoms, clinical diagnosis process, or follow-up outcomes.
- Any later prototype must be described as a learning and communication tool, not medical advice.

## 16. Final Checks

These checks confirm that the cleaned dataset and evidence files were created successfully.

In [ ]:
required_output_files = [
    PROCESSED_PATH,
    TABLES_DIR / 'cardio_cleaning_summary.csv',
    TABLES_DIR / 'cardio_target_balance_before_after_cleaning.csv',
    TABLES_DIR / 'cardio_eda_group_rate_summary.csv',
]

required_chart_files = [
    CHARTS_DIR / 'eda_01_target_distribution_after_cleaning.png',
    CHARTS_DIR / 'eda_02_rate_by_age_band.png',
    CHARTS_DIR / 'eda_03_rate_by_systolic_bp_category.png',
    CHARTS_DIR / 'eda_04_rate_by_diastolic_bp_category.png',
    CHARTS_DIR / 'eda_05_rate_by_bmi_category.png',
    CHARTS_DIR / 'eda_06_rate_by_cholesterol.png',
    CHARTS_DIR / 'eda_07_rate_by_glucose.png',
    CHARTS_DIR / 'eda_08_rate_by_activity.png',
    CHARTS_DIR / 'eda_09_numeric_correlation_heatmap.png',
]

missing_outputs = [path for path in [*required_output_files, *required_chart_files] if not path.exists()]

if missing_outputs:
    raise FileNotFoundError(f'Missing expected outputs: {missing_outputs}')

assert clean_df_out['cardio'].isin([0, 1]).all(), 'Target column should remain binary.'
assert clean_df_out.isna().sum().sum() == 0, 'Cleaned dataset should not contain missing values.'

print('Final checks passed.')
print(f'Cleaned rows: {len(clean_df_out):,}')
print(f'Charts saved in: {CHARTS_DIR}')
print(f'Tables saved in: {TABLES_DIR}')

## Next Step

After reviewing this notebook and the charts, the next notebook should build the logistic regression model using the cleaned dataset.